In [1]:
%pip install llama-index llama-index-embeddings-cohere llama-index-llms-cohere cohere python-dotenv
%pip install llama-index-vector-stores-pinecone pinecone-client
%pip install gradio

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from llama_cloud import TextNode
import os, httpx, ssl, urllib3
import gradio as gr
from dotenv import load_dotenv
from pinecone import Pinecone

# LlamaIndex Core & Components
from llama_index.core import (
    Settings, 
    StorageContext, 
    VectorStoreIndex, 
    SimpleDirectoryReader,
    get_response_synthesizer
)
from llama_index.embeddings.cohere import CohereEmbedding
from llama_index.llms.cohere import Cohere
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core.node_parser import MarkdownNodeParser
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.chat_engine import ContextChatEngine 
from llama_index.core.workflow import Event, StartEvent, StopEvent, Workflow, step
from llama_index.core import PromptTemplate


from pathlib import Path
from llama_index.utils.workflow import draw_all_possible_flows
from IPython.display import IFrame

import json
import time
from pydantic import BaseModel, Field
from typing import List
from llama_index.core.output_parsers import PydanticOutputParser
from llama_index.core.llms import ChatMessage



# --- NetFree & SSL Bypass ---
os.environ['CURL_CA_BUNDLE'] = ""
os.environ['HTTPX_VERIFY'] = "False" 
ssl._create_default_https_context = ssl._create_unverified_context
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

unsafe_client = httpx.Client(verify=False) 

print("✅ Step 2: All Imports and NetFree bypass configured.")

✅ Step 2: All Imports and NetFree bypass configured.


In [3]:
load_dotenv()

cohere_key = os.getenv("COHERE_API_KEY")

if not cohere_key:
    print("❌ Error: COHERE_API_KEY not found in .env file.")
else:
    print("✅ Step 3: API Keys loaded.")    

✅ Step 3: API Keys loaded.


In [4]:
# --- 1. Setup Cohere ---
Settings.embed_model = CohereEmbedding(
    api_key=os.getenv("COHERE_API_KEY"), 
    model_name="embed-multilingual-v3.0", 
    http_client=unsafe_client
)
Settings.llm = Cohere(
    api_key=os.getenv("COHERE_API_KEY"), 
    model="command-r-plus-08-2024",
)

# --- 2. Setup Node Parser ---
Settings.node_parser = MarkdownNodeParser()

# --- 3. Setup Pinecone ---
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"), ssl_verify=False)
pinecone_index = pc.Index("rag-project")
vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_vector_store(vector_store)

retriever = index.as_retriever(similarity_top_k=3)
response_synthesizer = get_response_synthesizer(llm=Settings.llm, response_mode="compact")
node_postprocessor = SimilarityPostprocessor(similarity_cutoff=0.2)

print("✅ Step 4: Models, Parser, Pinecone, and RAG Pipeline configured.")

✅ Step 4: Models, Parser, Pinecone, and RAG Pipeline configured.


In [5]:
print("✅ Step 5: Loading and Chunking documents...")

project_configs = {
    "Project_1": {"path": "../Data/cloude/project1", "type": "Cloude"},
    "Project_2": {"path": "../Data/cloude/project2", "type": "Cloude"},
    "Kiro_Logic": {"path": "../Data/kiro", "type": "Kiro"}
}

all_docs = []
print("--- Starting Data Loading ---")

for proj_id, config in project_configs.items():
    path = config["path"]
    tool_type = config["type"]
    
    try:
        if os.path.exists(path):
            documents = SimpleDirectoryReader(path).load_data()
            
            for doc in documents:
                doc.metadata["project_id"] = proj_id
                doc.metadata["tool_type"] = tool_type
                doc.metadata["file_name"] = doc.metadata.get("file_name", "unknown")
            
            all_docs.extend(documents)
            print(f"✅ {proj_id} ({tool_type}): Loaded {len(documents)} docs.")
        else:
            print(f"⚠️ Path not found: {path} (Skipping...)")
            
    except Exception as e:
        print(f"💥 Error in {proj_id}: {str(e)}")

if not all_docs:
    print("❌ Critical Error: No documents were loaded. Check your paths!")
else:
    print(f"\nTotal documents loaded: {len(all_docs)}")
    
    nodes = Settings.node_parser.get_nodes_from_documents(all_docs)
    print(f"✅ Created {len(nodes)} chunks (Nodes) using Markdown Strategy.")

✅ Step 5: Loading and Chunking documents...
--- Starting Data Loading ---
✅ Project_1 (Cloude): Loaded 3 docs.
✅ Project_2 (Cloude): Loaded 3 docs.
✅ Kiro_Logic (Kiro): Loaded 3 docs.

Total documents loaded: 9
✅ Created 213 chunks (Nodes) using Markdown Strategy.


In [6]:
# --- STEP 5.5: Structured Data Extraction ---

print("Starting Data Extraction process...")

# Define the schema using Pydantic
class Decision(BaseModel):
    title: str = Field(description="Short title of the technical decision")
    summary: str = Field(description="Summary of the decision in Hebrew")

class Rule(BaseModel):
    rule: str = Field(description="The rule or guideline in Hebrew")
    scope: str = Field(description="Scope of the rule (e.g., UI, DB, Backend)")

class WarningItem(BaseModel):
    message: str = Field(description="Warning or sensitive technical area in Hebrew")
    severity: str = Field(description="Severity level: high, medium, or low")

class ExtractedData(BaseModel):
    decisions: List[Decision] = Field(default_factory=list)
    rules: List[Rule] = Field(default_factory=list)
    warnings: List[WarningItem] = Field(default_factory=list)

# Initialize the Parser
parser = PydanticOutputParser(ExtractedData)
format_instructions = parser.get_format_string()

# Initialize the structured database structure
structured_db = {
    "schema_version": "1.0",
    "extracted_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "items": {"decisions": [], "rules": [], "warnings": []}
}

print("Scanning documents and extracting insights...")
print("Note: Adding a delay to respect API Rate Limits.\n")

for doc in all_docs:
    file_name = doc.metadata.get("file_name", "unknown")
    tool_type = doc.metadata.get("tool_type", "unknown")

    extraction_prompt = f"""
    Analyze the following text from the log file: {file_name}.
    Extract any technical decisions, rules/guidelines, and warnings.
    IMPORTANT: The summaries and descriptions MUST be in Hebrew.
    If no relevant items are found, return empty lists.

    {format_instructions}

    Text:
    {doc.text}
    """

    try:
        messages = [ChatMessage(role="user", content=extraction_prompt)]
        response = Settings.llm.chat(messages)
        
        # Parse the raw LLM response into the Pydantic model
        result = parser.parse(response.message.content)

        # Map results and add metadata
        for d in result.decisions:
            item = d.model_dump() # Using model_dump() for Pydantic v2
            item["source_file"] = file_name
            item["tool"] = tool_type
            structured_db["items"]["decisions"].append(item)

        for r in result.rules:
            item = r.model_dump()
            item["source_file"] = file_name
            item["tool"] = tool_type
            structured_db["items"]["rules"].append(item)

        for w in result.warnings:
            item = w.model_dump()
            item["source_file"] = file_name
            item["tool"] = tool_type
            structured_db["items"]["warnings"].append(item)

        print(f"Successfully processed: {file_name}")

    except Exception as e:
        print(f"Error processing {file_name}: {str(e)}")

    # Sleep to avoid 429 Rate Limit errors
    time.sleep(5)

# Save the final structured data to a JSON file
output_file = "structured_data.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(structured_db, f, ensure_ascii=False, indent=4)

print(f"\nExtraction complete! Data saved to {output_file}")

Starting Data Extraction process...
Scanning documents and extracting insights...
Note: Adding a delay to respect API Rate Limits.



2026-04-14 16:47:41,085 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"


Successfully processed: spec.md


2026-04-14 16:48:02,063 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"


Successfully processed: ui_rules.md


...

Successfully processed: patient_privacy.md


2026-04-14 16:50:43,543 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"


Successfully processed: scheduling_algorithm.md


2026-04-14 16:51:17,654 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"


Successfully processed: system_requirements.md

Extraction complete! Data saved to structured_data.json


In [17]:
print("✅ Step 6: Automatic Index Management...")

try:
    index_stats = pinecone_index.describe_index_stats()
    vector_count = index_stats.get('total_vector_count', 0)

    if vector_count == 0:
        print(f"🚀 Index is empty. Indexing {len(nodes)} nodes for the first time...")
        index = VectorStoreIndex(
            nodes, 
            storage_context=storage_context, 
            show_progress=True
        )
    else:
        print(f"🔗 Found {vector_count} existing vectors. Connecting to current index...")
        index = VectorStoreIndex.from_vector_store(
            vector_store, 
            storage_context=storage_context
        )

    retriever = index.as_retriever(similarity_top_k=5)
    print("✨ Index is ready for use!")

except Exception as e:
    print(f"❌ Error during index initialization: {e}")

✅ Step 6: Automatic Index Management...


2026-04-14 17:07:38,752 - WARNING - Retrying (JitterRetry(total=4, connect=None, read=None, redirect=None, status=None)) after connection broken by 'NameResolutionError("HTTPSConnection(host='rag-project-i0kvufd.svc.aped-4627-b74a.pinecone.io', port=443): Failed to resolve 'rag-project-i0kvufd.svc.aped-4627-b74a.pinecone.io' ([Errno 11002] getaddrinfo failed)")': /describe_index_stats
2026-04-14 17:07:39,353 - WARNING - Retrying (JitterRetry(total=3, connect=None, read=None, redirect=None, status=None)) after connection broken by 'NameResolutionError("HTTPSConnection(host='rag-project-i0kvufd.svc.aped-4627-b74a.pinecone.io', port=443): Failed to resolve 'rag-project-i0kvufd.svc.aped-4627-b74a.pinecone.io' ([Errno 11002] getaddrinfo failed)")': /describe_index_stats
2026-04-14 17:07:40,502 - WARNING - Retrying (JitterRetry(total=2, connect=None, read=None, redirect=None, status=None)) after connection broken by 'NameResolutionError("HTTPSConnection(host='rag-project-i0kvufd.svc.aped-462

🔗 Found 1591 existing vectors. Connecting to current index...
✨ Index is ready for use!


In [26]:
# --- Event Definitions ---

class RetrievalEvent(Event):
    query: str

class StructuredSearchEvent(Event):
    query: str
    category: str

class SynthesizeEvent(Event):
    query: str
    context: str

# --- Smart Agent Workflow ---

class SmartAgentWorkflow(Workflow):
    
    @step
    async def route_query(self, ev: StartEvent) -> RetrievalEvent | StructuredSearchEvent:
        query = ev.query
        print(f"🚀 AI Agent routing query: '{query}'")

        routing_prompt = f"""
        Analyze the user query: "{query}"
        Identify the intent and the category.
        Categories: 'rules', 'decisions', 'warnings', or 'none'.
        
        Respond ONLY with a JSON object like this:
        {{"route": "structured", "category": "rules"}} 
        OR 
        {{"route": "semantic", "category": "none"}}
        """
                
        message = ChatMessage(role="user", content=routing_prompt)
        response = Settings.llm.chat([message])
        clean_res = str(response.message.content).strip().replace("```json", "").replace("```", "")
        result = json.loads(clean_res)

        if result["route"] == "structured":
            print(f"🔀 Route: Structured ({result['category']})")
            return StructuredSearchEvent(query=query, category=result["category"])
        
        print("🔀 Route: Semantic RAG")
        return RetrievalEvent(query=query)

    @step
    async def handle_structured_search(self, ev: StructuredSearchEvent) -> SynthesizeEvent:
        print(f"📂 Searching JSON for category: {ev.category}")
        try:
            with open("structured_data.json", "r", encoding="utf-8") as f:
                data = json.load(f)
            
            items = []
            if ev.category in ["rules", "all"]: items.extend(data["items"].get("rules", []))
            if ev.category in ["decisions", "all"]: items.extend(data["items"].get("decisions", []))
            if ev.category in ["warnings", "all"]: items.extend(data["items"].get("warnings", []))
            
            return SynthesizeEvent(query=ev.query, context=str(items))
        except Exception as e:
            print(f"❌ JSON Search Error: {e}")
            return SynthesizeEvent(query=ev.query, context="No structured data available.")

    @step
    async def handle_retrieval(self, ev: RetrievalEvent) -> SynthesizeEvent:
        print("🔍 Searching Pinecone vector store...")
        nodes = retriever.retrieve(ev.query)
        context = "\n\n".join([n.node.get_content() for n in nodes])
        return SynthesizeEvent(query=ev.query, context=context)

    @step
    async def synthesize_response(self, ev: SynthesizeEvent) -> StopEvent:
        print("✍️ Generating final professional answer...")
        
        # חשוב לייבא את האובייקטים האלו
        from llama_index.core.schema import TextNode, NodeWithScore

        # יצירת המבנה שהסינתסייזר "אוהב"
        wrapped_node = NodeWithScore(
            node=TextNode(text=ev.context),
            score=1.0
        )
        
        # עכשיו ה-Synthesizer ימצא את n.node ולא יקרוס
        response = response_synthesizer.synthesize(
            query=ev.query,
            nodes=[wrapped_node] 
        )
        
        return StopEvent(result=str(response))

In [27]:
retriever = index.as_retriever(similarity_top_k=12)
node_postprocessor = SimilarityPostprocessor(similarity_cutoff=0.12)

response_synthesizer = get_response_synthesizer(
    response_mode="tree_summarize", 
    summary_template=PromptTemplate(
        "You are a professional AI assistant for developers.\n"
        "IMPORTANT: Always respond in the same language as the user's question.\n"
        "Context information is below:\n"
        "---------------------\n"
        "{context_str}\n"
        "---------------------\n"
        "Instructions:\n"
        "1. Project 1: Task Management (React, Vite, SQLite).\n"
        "2. Project 2: Logistics/Inventory (Azure, SQL, Purchase Orders).\n"
        "3. Kiro Logic/Project 3: Medical/Clinic (HIPAA, Security, Encryption).\n"
        "If a project like Project 4 or 5 is mentioned and NOT in the context, say you don't know.\n"
        "Query: {query_str}\n"
        "Answer:"
    )
)

In [28]:
# --- STEP 8: Initialize Workflow and Bridge Function ---

agent_workflow = SmartAgentWorkflow(timeout=120)

async def predict_workflow(message, history):
    """
    Bridge function to connect Gradio to our SmartAgentWorkflow.
    """
    try:
        # We call the run method with the user's message
        print(f"🚀 Agent is processing: {message}")
        result = await agent_workflow.run(query=message)
        return str(result)
        
    except Exception as e:
        print(f"🔴 Workflow Error: {e}")
        return f"❌ System Error: {str(e)}"

print("✅ Step 8: Smart Agent Workflow initialized and bridge function ready.")

✅ Step 8: Smart Agent Workflow initialized and bridge function ready.


In [29]:
Path("workflows").mkdir(parents=True, exist_ok=True)

draw_all_possible_flows(
    agent_workflow,  
    filename=str(Path("workflows/rag_workflow.html").resolve())
)

IFrame(src='./workflows/rag_workflow.html', width=1000, height=800)

C:\Users\user1\Desktop\miri\תכנות\AI\פרויקטים\RagProject\Project\workflows\rag_workflow.html


In [30]:
custom_css = """
.gradio-container {
    max-width: 100% !important; 
    width: 100% !important;
    margin: 0 !important;
    padding: 20px !important;
    direction: rtl; 
}
#chatbot {
    height: 750px !important; 
    width: 100% !important;
    border-radius: 15px;
    box-shadow: 0 4px 25px rgba(0,0,0,0.1);
}
.message-wrap .message-text {
    text-align: right;
    direction: rtl;
}
@keyframes fadeInMsg { 
    from { opacity: 0; transform: translateY(10px); } 
    to { opacity: 1; transform: translateY(0); } 
}
.message-row { animation: fadeInMsg 0.3s ease-out; }
"""

theme = gr.themes.Soft(
    primary_hue="blue", 
    neutral_hue="zinc", 
    font=[gr.themes.GoogleFont("Assistant"), "sans-serif"] 
)

with gr.Blocks(title="AI Developer Assistant", fill_width=True) as demo:
    gr.Markdown("# 🤖 AI Agentic Coding Assistant (Workflow Mode)")
    gr.Markdown("I can now use **tools** to analyze your projects more deeply.")
    gr.Markdown("---") 

    gr.ChatInterface(
        fn=predict_workflow, 
        chatbot=gr.Chatbot(elem_id="chatbot", show_label=False),
        textbox=gr.Textbox(
            placeholder="Ask me to analyze or do something with your logs...",
            scale=7,
            show_label=False
        ),
        submit_btn="Send 🚀",
        examples=[
            "Summarize the main tasks in Project 1 using the Search Tool.",
            "Compare the security between Project 2 and Kiro.",
            "What are the UI decisions for the Task Manager?"
        ]
    )

if __name__ == "__main__":
    demo.launch(
        share=True, 
        theme=theme, 
        css=custom_css
    )

2026-04-14 17:19:19,063 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
2026-04-14 17:19:19,164 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
C:\Users\user1\AppData\Local\Temp\ipykernel_14776\342524718.py:37: UserWarning: You provided a custom `textbox` component, but also specified `submit_btn` parameter(s) on `gr.ChatInterface`. These ChatInterface parameters will be ignored. To customize these settings, pass them directly to your `gr.Textbox` or `gr.MultimodalTextbox` component instead. For example: textbox=gr.Textbox(..., submit_btn='submit')
  gr.ChatInterface(


* Running on local URL:  http://127.0.0.1:7864


2026-04-14 17:19:20,130 - INFO - HTTP Request: GET http://127.0.0.1:7864/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-04-14 17:19:20,153 - INFO - HTTP Request: HEAD http://127.0.0.1:7864/ "HTTP/1.1 200 OK"
2026-04-14 17:19:20,684 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-04-14 17:19:20,711 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-04-14 17:19:21,813 - INFO - HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"
2026-04-14 17:19:22,996 - INFO - HTTP Request: GET https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_windows_amd64.exe "HTTP/1.1 200 OK"



Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026-04-14 17:19:24,988 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-error-analytics "HTTP/1.1 200 OK"
2026-04-14 17:19:25,069 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"


...

2026-04-14 17:26:13,558 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"


🚀 Agent is processing: תן לי את כל הכללים שבכל הפרויקטים שנקבעו
🚀 AI Agent routing query: 'תן לי את כל הכללים שבכל הפרויקטים שנקבעו'


2026-04-14 17:27:40,566 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"


🔀 Route: Structured (rules)
📂 Searching JSON for category: rules
✍️ Generating final professional answer...


2026-04-14 17:28:04,157 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"
